# Dashboard

The "Latest and Greatest" page of the old Power BI report, rebuilt with Altair from
`processed_data.processed_responses` (made by `processing/new_proccess_data.ipynb`). The subtitle also
shows when each `static_data` table was last fetched, from `static_data.fetched`.

The whole page is one Altair chart, so its filters run in the browser, also in the saved
`dashboard.html`:

- **All / Identified / Verified / Unique visitors** (top right): count every visit, visits with a student number,
  visits by students found in the student data, or each student once per bar.
- **Dates**: drag across the timeline; click the timeline outside the selection to remove it.
- **Semester, education, study start, gender, location, study line, AI use, department, course**: click a bar or slice, shift-click to pick multiple.
  The other charts filter; the clicked one highlights. Click a picked bar again, or empty space in its chart, to
  remove it.
- **Clear filters** (top right) removes every filter.

Staff visits are left out: the dashboard covers students. Anonymous visits (no student number) count only
under All visitors.

In [ ]:
import json
import os
from pathlib import Path

import altair as alt
import pandas as pd
from pis_lib.connect import get_pg_engine

alt.data_transformers.disable_max_rows()  # the default 5000-row guard; every row is embedded in the page

In [ ]:
pg = get_pg_engine()

responses_df = pd.read_sql(
    'SELECT created_at, visitor_id, satisfaction, building_number, gender, "education_EN", "direction_DK",'
    " weekday, time_slot, semester, study_start, verified, ai, course, department"
    " FROM processed_data.processed_responses WHERE NOT staff",
    pg,
)
fetched = pd.read_sql("SELECT table_name, fetched_at FROM static_data.fetched ORDER BY 1", pg)

## Prepare

- `week` and `month` are the start of each period, in Copenhagen time without a time zone, so every
  browser shows the same dates.
- Charts colored by education show the 5 largest groups; the rest are `Other`, so no chart needs more
  than the 8 palette colors.
- Missing student data becomes `Unknown`. `visitor_id` stays NULL: those visits have no student.
- `verified` becomes 1 or 0, so charts can add it up.
- `FETCHED` lists the day each `static_data` table was last fetched, in Copenhagen time.
- `AI_SINCE` and `COURSE_SINCE`, the first visits with an `ai` or a `course` value, are when each question
  was added. `course` keeps the 15 courses named most often; other visits get no value.
- `department` is the department of the course named, without `Department of`: `01 Applied Mathematics and
  Computer Science`. `Unknown` when the course data has none; no value for visits that named no course.

In [ ]:
UNKNOWN = "Unknown"
NO_ANSWER = "No answer"
AI_ANSWERS = ["Yes", "No", NO_ANSWER]
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

local_time = responses_df["created_at"].dt.tz_convert("Europe/Copenhagen").dt.tz_localize(None)
top_educations = responses_df["education_EN"].value_counts().index[:5]
AI_SINCE = local_time[responses_df["ai"].notna()].min()
COURSE_SINCE = local_time[responses_df["course"].notna()].min()
top_courses = responses_df["course"].value_counts().index[:15]

df = (
    responses_df
    .assign(
        week=local_time.dt.to_period("W").dt.start_time,
        month=local_time.dt.to_period("M").dt.start_time,
        education_group=responses_df["education_EN"].where(
            responses_df["education_EN"].isin(top_educations) | responses_df["education_EN"].isna(), "Other"
        ),
        verified=responses_df["verified"].astype(int),
        course=responses_df["course"].where(responses_df["course"].isin(top_courses)),
        department=responses_df["department"].str.replace(" Department of ", " ", regex=False)
        .fillna(UNKNOWN).where(responses_df["course"].notna()),
    )
    .drop(columns="created_at")
    .fillna({column: UNKNOWN for column in ["gender", "education_EN", "education_group", "direction_DK", "study_start"]})
)


def order(column, first=None):
    """The column's values in a fixed order (`first`, else sorted), Unknown last."""
    present = set(df[column])
    ordered = [value for value in (sorted(present) if first is None else first) if value in present and value != UNKNOWN]
    return ordered + ([UNKNOWN] if UNKNOWN in present else [])


SEMESTERS = order("semester", list(df.sort_values("month")["semester"].unique()))
EDUCATIONS = order("education_EN", list(df["education_EN"].value_counts().index))
COURSES = list(top_courses)
DEPARTMENTS = order("department", list(df["department"].value_counts().index))
FETCHED = " · ".join(
    f"{row.table_name} {row.fetched_at.tz_convert('Europe/Copenhagen'):%-d %b %Y}" for row in fetched.itertuples()
) or "not recorded"

## Style

Colors follow the category, not its rank: each color scale lists every value, so a filter never
repaints the bars that remain. `Unknown` and `No answer` are always gray.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"


def color(column, title, values=None):
    domain = values or order(column)  # Unknown or No answer, if present, is last
    known = [value for value in domain if value not in (UNKNOWN, NO_ANSWER)]
    assert len(known) <= len(PALETTE), f"{column} has more values than colors; fold the rest into Other"
    return alt.Color(
        f"{column}:N", title=title, sort=domain,
        scale=alt.Scale(domain=domain, range=PALETTE[: len(known)] + [MUTED] * (len(domain) - len(known))),
    )

## Filters

Every chart filters by all the selections except its own; its own one highlights instead, as in Power BI.
Every filter clears when the Clear filters button is clicked (a click on any mark whose data has
`action: "clear"`). The toggle is a selection too, which never clears: `MODE` is the picked button,
"All visitors" if none is.

In [ ]:
CLEAR = "click[event.item && event.item.datum && event.item.datum.action == 'clear']"


def pick(name, *fields):
    """A click filter: clicking a picked value again removes it (toggles), shift-click adds or removes one."""
    return alt.selection_point(
        fields=list(fields), clear=CLEAR, name=name,
        toggle=f"event.shiftKey || (datum && vlSelectionTest('{name}_store', datum))",
    )


dates = alt.selection_interval(encodings=["x"], clear=CLEAR, name="dates")
semester_pick = pick("semester_pick", "semester")
education_pick = pick("education_pick", "education_EN")
start_pick = pick("start_pick", "study_start")
gender_pick = pick("gender_pick", "gender")
location_pick = pick("location_pick", "building_number")
line_pick = pick("line_pick", "education_EN", "direction_DK")  # one bar segment
ai_pick = pick("ai_pick", "ai")
department_pick = pick("department_pick", "department")
course_pick = pick("course_pick", "course")
FILTERS = [dates, semester_pick, education_pick, start_pick, gender_pick, location_pick, line_pick, ai_pick,
           department_pick, course_pick]

visitor_mode = alt.selection_point(
    fields=["mode"], value=[{"mode": "All visitors"}], toggle=False, clear=False, name="visitor_mode"
)
MODES = {  # button: what it counts, shown on hover
    "All visitors": "Every visit, including anonymous ones. A student who comes back counts each time.",
    "Identified visitors": "Visits with a student number. A student who comes back counts each time.",
    "Verified visitors": "Visits by students found in the student data. A student who comes back counts each time.",
    "Unique visitors": "Each student once per bar or slice. Anonymous visits are left out.",
}
MODE = "((visitor_mode.mode && visitor_mode.mode[0]) || 'All visitors')"

base = alt.Chart(df)
ai_base = base.transform_filter("isValid(datum.ai)")
CLICK = "Click to filter, again to remove, shift-click for multiple"


def visitors(chart, by, own=None):
    """Filter by every selection except `own`, then count per `by`.

    `visitors` is what the toggle picks: all `visits`, `identified` visits (with a student number), `verified`
    visits (student found in the student data), or `unique` students.
    """
    for selection in FILTERS:
        if selection is not own:
            chart = chart.transform_filter(selection)
    return chart.transform_aggregate(
        visits="count()", students="distinct(visitor_id)", anonymous="missing(visitor_id)", verified="sum(verified)",
        rating="mean(satisfaction)", groupby=by,
    ).transform_calculate(
        identified="datum.visits - datum.anonymous",
        unique="datum.students - (datum.anonymous > 0 ? 1 : 0)",  # distinct() counts null as one more value
        visitors=f"{MODE} == 'Unique visitors' ? datum.unique : {MODE} == 'Verified visitors' ? datum.verified"
                 f" : {MODE} == 'Identified visitors' ? datum.identified : datum.visits",
    )


def tooltip(*fields):
    return [*fields, alt.Tooltip("visits:Q", title="All visitors"), alt.Tooltip("identified:Q", title="Identified visitors"),
            alt.Tooltip("verified:Q", title="Verified visitors"), alt.Tooltip("unique:Q", title="Unique visitors"),
            alt.Tooltip("rating:Q", title="Average rating", format=".2f")]


def highlight(selection):
    return alt.condition(selection, alt.value(1), alt.value(0.3))

## Charts

In [ ]:
def kpi(title, field, number_format):
    return visitors(base, []).mark_text(fontSize=30, align="left", color=INK).encode(
        x=alt.value(0), text=alt.Text(f"{field}:Q", format=number_format),
    ).properties(title=alt.Title(title, fontSize=12, color=INK_2, fontWeight="normal"), width=110, height=36)


kpis = alt.hconcat(kpi("Average rating", "rating", ".2f"), kpi("All visitors", "visits", ","),
                   kpi("Identified visitors", "identified", ","), kpi("Verified visitors", "verified", ","),
                   kpi("Unique visitors", "unique", ","))

toggle_base = alt.Chart(alt.InlineData(values=[{"mode": m, "description": d} for m, d in MODES.items()])).encode(
    x=alt.X("mode:N", axis=None, sort=list(MODES)),
)
picked = f"{MODE} == datum.mode"  # this button's mode is the one in use
toggle = alt.layer(
    toggle_base.mark_rect(cornerRadius=6, stroke=SURFACE, strokeWidth=6, cursor="pointer").encode(
        color=alt.condition(picked, alt.value(PALETTE[0]), alt.value(GRID)),
        tooltip="description:N",
    ).add_params(visitor_mode),
    toggle_base.mark_text(fontSize=13, cursor="pointer").encode(
        text="mode:N", color=alt.condition(picked, alt.value("white"), alt.value(INK_2)),
    ),
).properties(width=125 * len(MODES), height=40, title=alt.Title("Count", fontSize=12, color=INK_2, fontWeight="normal"))

# The button counts the filters in use, so it turns blue while any are on and gray once a click clears them.
active_filters = " + ".join(f"(length(data('{selection.name}_store')) > 0 ? 1 : 0)" for selection in FILTERS)
clear_base = alt.Chart(alt.InlineData(values=[{"action": "clear"}])).transform_calculate(
    active=active_filters,
    label="datum.active ? 'Clear ' + datum.active + (datum.active == 1 ? ' filter' : ' filters') : 'No filters'",
).encode(x=alt.X("action:N", axis=None))
clear_button = alt.layer(
    clear_base.mark_rect(cornerRadius=6, stroke=SURFACE, strokeWidth=6, cursor="pointer").encode(
        color=alt.condition("datum.active > 0", alt.value(PALETTE[0]), alt.value(GRID)),
        # Vega-Lite only lets marks with a selection or a tooltip receive clicks; without it clicks fall through.
        tooltip=alt.value("Remove every filter"),
    ),
    clear_base.mark_text(fontSize=13, cursor="pointer").encode(
        text="label:N", color=alt.condition("datum.active > 0", alt.value("white"), alt.value(MUTED)),
    ),
).properties(width=130, height=40, title=alt.Title("Filters", fontSize=12, color=INK_2, fontWeight="normal"))

# A fixed date range: Vega-Lite's brush breaks ("dates_week is undefined") when its axis rescales with no brush drawn.
# With x2, Vega-Lite then logs 'Conflicting scale property "domain"'; harmless, both domains are this one.
first_week, last_week = df["week"].min(), df["week"].max() + pd.Timedelta(weeks=1)
timeline = visitors(base, ["week"], own=dates).transform_calculate(
    week_end="timeOffset('week', datum.week, 1)",
).mark_rect(color=PALETTE[0], stroke=SURFACE, strokeWidth=0.5).encode(
    x=alt.X("week:T", title=None, scale=alt.Scale(domain=[
        alt.DateTime(year=d.year, month=d.month, date=d.day) for d in (first_week, last_week)
    ])),
    x2="week_end:T",
    y=alt.Y("visitors:Q", title=None), y2=alt.datum(0),
    tooltip=tooltip(alt.Tooltip("week:T", title="Week of", format="%-d %b %Y")),
).add_params(dates).properties(
    width=560, height=120, title=alt.Title("Dates", subtitle="Drag to filter, click outside the selection to remove it"),
)


def bars(column, pick, title, values, width=200, chart=base, subtitle=CLICK):
    """Horizontal bars per category that double as a slicer: click one to filter the other charts."""
    return visitors(chart, [column], own=pick).mark_bar(color=PALETTE[0], cornerRadiusEnd=4, cursor="pointer").encode(
        y=alt.Y(f"{column}:N", sort=values, title=None),
        x=alt.X("visitors:Q", title=None),
        opacity=highlight(pick),
        tooltip=tooltip(alt.Tooltip(f"{column}:N", title=title)),
    ).add_params(pick).properties(width=width, height=alt.Step(18), title=alt.Title(title, subtitle=subtitle))


def pie(column, pick, title, values=None, chart=base, subtitle=CLICK):
    return visitors(chart, [column], own=pick).mark_arc(innerRadius=45, stroke=SURFACE, strokeWidth=2, cursor="pointer").encode(
        theta=alt.Theta("visitors:Q", stack=True),
        color=color(column, title, values),
        opacity=highlight(pick),
        tooltip=tooltip(alt.Tooltip(f"{column}:N", title=title)),
    ).add_params(pick).properties(width=180, height=180, title=alt.Title(title, subtitle=subtitle))


def by_gender(column, column_title, values, stack, stack_title, stack_values, title):
    """Columns per gender, grouped under each `column` value and stacked by `stack`."""
    return visitors(base, [column, "gender", stack]).mark_bar(stroke=SURFACE, strokeWidth=2).encode(
        column=alt.Column(f"{column}:N", sort=values, title=None, spacing=6,
                          header=alt.Header(labelOrient="bottom", labelColor=INK_2, labelFontSize=11)),
        x=alt.X("gender:N", sort=order("gender"), title=None, axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("visitors:Q", title="Visitors"),
        color=color(stack, stack_title, stack_values),
        tooltip=tooltip(alt.Tooltip(f"{column}:N", title=column_title), alt.Tooltip("gender:N", title="Gender"),
                        alt.Tooltip(f"{stack}:N", title=stack_title)),
    ).properties(width=alt.Step(16), height=220, title=title)


weekday_chart = by_gender("weekday", "Weekday", order("weekday", WEEKDAYS), "time_slot", "Time slot",
                          ["Morning", "Afternoon"], "Visitors by weekday and gender")
start_chart = by_gender("study_start", "Study start", order("study_start"), "education_group", "Education",
                        order("education_group"), "Visitors by study start and gender")

kinds = ["Unique visitors", "Repeat visits", "Anonymous visits"]
returning_chart = visitors(base, ["month"]).transform_calculate(
    **{"Unique visitors": "datum.unique", "Repeat visits": "datum.visits - datum.unique - datum.anonymous",
       "Anonymous visits": "datum.anonymous"},
).transform_fold(kinds, as_=["kind", "count"]).mark_bar(stroke=SURFACE, strokeWidth=1).encode(
    x=alt.X("yearmonth(month):T", title=None),
    y=alt.Y("count:Q", stack="normalize", title=None, axis=alt.Axis(format="%")),
    color=alt.Color("kind:N", title=None, sort=kinds, scale=alt.Scale(domain=kinds, range=[PALETTE[0], PALETTE[1], MUTED])),
    order=alt.Order("kind_order:Q"),
    tooltip=[alt.Tooltip("yearmonth(month):T", title="Month"), alt.Tooltip("kind:N", title=" "), alt.Tooltip("count:Q", title="Visits")],
).transform_calculate(kind_order=f"indexof({kinds}, datum.kind)").properties(
    width=460, height=200,
    title=alt.Title("Unique, repeat and anonymous visits by month", subtitle="Repeat: a student's 2nd or later visit that month"),
)

# As in Power BI: a running sum of each month's count, so a student who comes in two months counts twice.
cumulative = visitors(base, ["semester", "month"]).transform_window(
    so_far="sum(visitors)", groupby=["semester"], sort=[alt.SortField("month")],
).transform_calculate(
    season="split(datum.semester, ' ')[0]",
).mark_bar(cornerRadiusEnd=4, stroke=SURFACE, strokeWidth=2).encode(
    x=alt.X("yearmonth(month):T", title=None),
    y=alt.Y("so_far:Q", title="Visitors so far"),
    color=alt.Color("season:N", title="Semester", scale=alt.Scale(domain=["Spring", "Fall"], range=PALETTE[:2])),
    tooltip=[alt.Tooltip("semester:N", title="Semester"), alt.Tooltip("yearmonth(month):T", title="Month"),
             alt.Tooltip("visitors:Q", title="This month"), alt.Tooltip("so_far:Q", title="Semester so far")],
).properties(width=1000, height=220, title=alt.Title("Cumulative visitors per semester", subtitle="Adds up each month's count; restarts every semester"))

education_chart = visitors(
    base.transform_filter(f"datum.education_EN != '{UNKNOWN}'"), ["education_group", "education_EN", "direction_DK"],
    own=line_pick,
).mark_bar(stroke=SURFACE, strokeWidth=1, cursor="pointer").encode(
    x=alt.X("direction_DK:N", sort="-y", title=None, axis=alt.Axis(labelAngle=-60, labelLimit=140)),
    y=alt.Y("visitors:Q", title="Visitors"),
    color=color("education_group", "Education"),
    opacity=highlight(line_pick),
    tooltip=tooltip(alt.Tooltip("education_EN:N", title="Education"), alt.Tooltip("direction_DK:N", title="Study line")),
).add_params(line_pick).properties(
    width=alt.Step(12), height=220,
    title=alt.Title("Visitors by study line and education", subtitle=f"Students found in the student data. {CLICK}"),
)

AI_TITLE = "Tried fixing with AI before"
ai_since = f"Since the question was added, {AI_SINCE:%-d %B %Y}." if pd.notna(AI_SINCE) else "No answers yet."
ai_chart = pie("ai", ai_pick, AI_TITLE, AI_ANSWERS, ai_base, f"{ai_since} {CLICK}")

course_since = f"since the question was added, {COURSE_SINCE:%-d %B %Y}" if pd.notna(COURSE_SINCE) else "no answers yet"
course_bars = bars("course", course_pick, "Course", COURSES, chart=base.transform_filter("isValid(datum.course)"),
                   subtitle=[f"The 15 courses named most often, {course_since}", CLICK])
department_bars = bars("department", department_pick, "Department", DEPARTMENTS, width=160,
                       chart=base.transform_filter("isValid(datum.department)"),
                       subtitle=[f"Of the course named, {course_since}", CLICK])

# Only highlights the course picked in course_bars, the one course slicer.
ai_course_chart = visitors(ai_base.transform_filter("isValid(datum.course)"), ["course", "ai"], own=course_pick).transform_joinaggregate(
    course_visitors="sum(visitors)", groupby=["course"],
).transform_calculate(
    share="datum.visitors / datum.course_visitors", ai_order=f"indexof({AI_ANSWERS}, datum.ai)",
).mark_bar(stroke=SURFACE, strokeWidth=2).encode(
    # Labels slant right: names hanging left of the plot would push every other row right.
    x=alt.X("course:N", sort=COURSES, title=None, axis=alt.Axis(labelAngle=45, labelLimit=180)),
    y=alt.Y("visitors:Q", title="Visitors"),
    color=color("ai", AI_TITLE, AI_ANSWERS),
    order=alt.Order("ai_order:Q"),
    opacity=highlight(course_pick),
    tooltip=tooltip(alt.Tooltip("course:N", title="Course"), alt.Tooltip("ai:N", title=AI_TITLE),
                    alt.Tooltip("share:Q", title="Share of the course", format=".0%")),
).properties(
    width=alt.Step(28), height=220,
    title=alt.Title("AI use by course", subtitle=f"The 15 courses named most often. {ai_since}"),
)

## Dashboard

Each chart keeps its own colors and legend (`independent`); by default Altair merges them into one.

In [ ]:
def row(*charts):
    return alt.hconcat(*charts, spacing=30).resolve_scale(color="independent")


dashboard = alt.vconcat(
    row(kpis, toggle, clear_button),
    row(timeline, bars("semester", semester_pick, "Semester", SEMESTERS, width=160),
        bars("education_EN", education_pick, "Education", EDUCATIONS)),
    row(weekday_chart, start_chart, pie("gender", gender_pick, "Gender")),
    row(pie("building_number", location_pick, "Location"), bars("study_start", start_pick, "Study start", order("study_start")),
        returning_chart),
    cumulative,
    row(education_chart, ai_chart),
    row(ai_course_chart, department_bars, course_bars),  # course_bars' labels, first in a row, would push every row right
    spacing=30,
).resolve_scale(color="independent").properties(
    title=alt.Title("Python Installation Support statistics", fontSize=20, subtitle=[
        f"{len(df):,} visits, {df['month'].min():%B %Y} to {df['month'].max():%B %Y} · "
        # Old during opening hours (hourly runs) means the pipeline is failing.
        f"built {pd.Timestamp.now(tz='Europe/Copenhagen'):%d %b %Y %H:%M}",
        f"Data fetched: {FETCHED}",
    ]),
).configure(
    background=SURFACE, font="system-ui, -apple-system, 'Segoe UI', sans-serif",
).configure_view(stroke=None).configure_title(
    anchor="start", color=INK, subtitleColor=MUTED, fontSize=14, fontWeight=600,
).configure_axis(
    domainColor=AXIS, tickColor=AXIS, gridColor=GRID, labelColor=MUTED, titleColor=INK_2, titleFontWeight="normal",
).configure_legend(labelColor=INK_2, titleColor=INK_2, titleFontWeight="normal")

dashboard

Save the page. It embeds every row of `df` (no student numbers or names), so share it only with people who
may see the survey data, and don't commit it.

`DASHBOARD_HTML` sets where it goes instead. On the server, the pipeline saves it outside the checkout, and the
admin shows it at `/admin/dashboard/`.

In [ ]:
class ScriptSafeJSON(json.JSONEncoder):
    """JSON that can't close the page's <script> tag: every < becomes \\u003c, which reads back the same."""

    def encode(self, o):
        return super().encode(o).replace("<", "\\u003c")


page = Path(os.environ.get("DASHBOARD_HTML", "dashboard.html"))
partial = page.with_suffix(".tmp.html")  # still .html, so git ignores it too
dashboard.save(
    partial,
    inline=True,  # the page carries its own copy of Vega instead of loading it from a CDN
    json_kwds={"cls": ScriptSafeJSON},  # a "</script>" in the data can't break the page
    embed_options={
        # Vega runs the chart's expressions with its interpreter instead of eval, which the admin's
        # Content-Security-Policy for this page blocks.
        "ast": True,
        # The ⋯ menu offers only image downloads: "Open in Vega Editor" would send every row to vega.github.io.
        "actions": {"export": True, "source": False, "compiled": False, "editor": False},
    },
)
partial.replace(page)  # in one step, so nobody ever opens a half-written page